# Exercise: Implementing a User-Facing LLM Engine

Welcome! In this exercise, you'll build a simplified version of a user-facing LLM engine, similar in concept to vLLM's `LLMEngine`. The goal is to create a class that can accept user prompts and return the generated text as a stream of tokens. This is a common pattern in modern LLM serving systems that allows for a much better user experience, as users can see the output as it's being generated.

You will be working with a mock "core" engine (`MiniEngineCore`) that simulates the complex scheduling and batching happening under the hood. Your task is to build the user-friendly API on top of it.

**In this exercise you will:**
*   Implement a `UserLLMEngine` class to manage and track user requests.
*   Create a method to add new generation requests to the core engine.
*   Build an asynchronous `generate` method that returns a streaming iterator for the output tokens.

Let's get started!

In [ ]:
import asyncio
import uuid
import time
from collections import deque
from dataclasses import dataclass, field
from typing import Dict, List, AsyncIterator, Optional

# --- Helper Code (Students should NOT modify this) ---

@dataclass
class RequestOutput:
    """Output of a single step for a single request."""
    request_id: str
    token: str
    finished: bool = False

@dataclass
class EngineRequest:
    """A request tracked internally by the core engine."""
    request_id: str
    prompt: str
    # A mapping from prompt to a deterministic sequence of tokens
    _token_iterator: AsyncIterator = field(init=False)
    
    # This is a mock generation process. In a real system, this would
    # be the model generating tokens one by one.
    MOCK_GENERATION_SEQUENCES = {
        "Hello": [" My", " name", " is", " AI", "."],
        "The capital of France is": [" Par", "is", "."],
    }
    EOS_TOKEN = "[EOS]" # End of Sequence token

    def __post_init__(self):
        tokens = self.MOCK_GENERATION_SEQUENCES.get(self.prompt, ["Invalid", " prompt", "."])
        tokens.append(self.EOS_TOKEN)
        
        async def token_generator():
            for token in tokens:
                yield token
        
        self._token_iterator = token_generator()

    async def get_next_token(self) -> Optional[str]:
        try:
            return await self._token_iterator.__anext__()
        except StopAsyncIteration:
            return None

class MiniEngineCore:
    """
    A simplified, mock version of a low-level LLM engine scheduler.
    
    This class simulates the core scheduling loop. In each `step`, it
    processes active requests and generates one new token for each. Your
    `UserLLMEngine` will interact with this class.
    """
    def __init__(self):
        self._requests_to_add: deque[EngineRequest] = deque()
        self._active_requests: Dict[str, EngineRequest] = {}
        self.output_queue: deque[RequestOutput] = deque()

    def add_request(self, request_id: str, prompt: str):
        """Adds a new request to be processed in the next step."""
        self._requests_to_add.append(EngineRequest(request_id, prompt))
        print(f"CORE: Queued request {request_id[:6]} for prompt: '{prompt}'")

    async def step(self) -> List[RequestOutput]:
        """
        Simulates one iteration of the engine's scheduling loop.
        - Adds new requests from the queue.
        - "Generates" one token for each active request.
        - Returns a list of all outputs produced in this step.
        """
        # 1. Add new requests
        while self._requests_to_add:
            req = self._requests_to_add.popleft()
            self._active_requests[req.request_id] = req

        if not self._active_requests:
            return []

        # 2. Process active requests
        newly_finished_reqs = []
        step_outputs = []
        
        # In a real engine, this part is highly complex (batching, paging, etc.)
        # Here, we just generate one token for every active request.
        for request_id, req in self._active_requests.items():
            token = await req.get_next_token()
            if token is None:
                continue # Should not happen with our EOS logic
            
            finished = (token == EngineRequest.EOS_TOKEN)
            output = RequestOutput(request_id=request_id, token=token, finished=finished)
            step_outputs.append(output)

            if finished:
                newly_finished_reqs.append(request_id)
        
        # 3. Clean up finished requests
        for request_id in newly_finished_reqs:
            del self._active_requests[request_id]
            print(f"CORE: Finished request {request_id[:6]}")
            
        # Simulate processing time
        await asyncio.sleep(0.01)

        return step_outputs

print("Setup complete. MiniEngineCore and helper classes are defined.")

### Exercise 1: `UserLLMEngine` Initialization and Request Management

Your first task is to begin implementing the `UserLLMEngine`. This class will be the main interface for users. You need to implement two methods:

1.  `__init__(self, engine_core: MiniEngineCore)`: The constructor should initialize the engine. It needs to hold a reference to the `MiniEngineCore` and set up a data structure to track the requests it's managing. A dictionary is a great choice for this.

2.  `add_request(self, prompt: str) -> str`: This method is responsible for creating a new request. It should:
    *   Generate a unique ID for the request (we'll use `uuid`).
    *   Store some information about the request internally. For now, just storing the prompt associated with the ID is enough.
    *   Crucially, it must add the request to the `MiniEngineCore` so it can be processed.
    *   Return the unique request ID to the caller.

**Hint:** To store request information, you can use a dictionary where keys are the request IDs and values are the prompts or other relevant data.

In [ ]:
# GRADED FUNCTION: UserLLMEngine (Part 1)

@dataclass
class UserRequestState:
    """State of a single user request being tracked by UserLLMEngine."""
    prompt: str
    # You could add timestamps, sampling_params, etc. here
    
class UserLLMEngine:
    """
    A user-facing API for the LLM engine. It handles request submission
    and streams back the results.
    """
    def __init__(self, engine_core: MiniEngineCore):
        """
        Initializes the UserLLMEngine.
        
        Args:
            engine_core: An instance of MiniEngineCore that does the actual work.
        """
        self.engine_core = engine_core
        # A dictionary to track active requests managed by this UserLLMEngine
        self.requests: Dict[str, UserRequestState] = {}
        # A queue to hold outputs from the core that are ready for consumption
        self.output_queue: deque[RequestOutput] = deque()

    def add_request(self, prompt: str) -> str:
        """
        Adds a new generation request to the system.
        
        Args:
            prompt: The user's prompt.
            
        Returns:
            A unique request ID for this request.
        """
        ### START CODE HERE ###
    # your code here
    pass
        ### END CODE HERE ###

    async def generate(self, prompt: str) -> AsyncIterator[str]:
        """(You will implement this in Exercise 2)"""
        # This is a placeholder. You'll implement the real logic in the next step.
        if False:
            yield

In [ ]:
# Test for Exercise 1

async def test_add_request():
    print("--- Running Test for Exercise 1 ---")
    core = MiniEngineCore()
    engine = UserLLMEngine(engine_core=core)

    # Test adding a single request
    prompt1 = "Hello"
    request_id1 = engine.add_request(prompt1)

    assert isinstance(request_id1, str) and request_id1, "add_request should return a non-empty string request ID."
    assert request_id1 in engine.requests, f"Request ID {request_id1} not found in engine.requests."
    assert engine.requests[request_id1].prompt == prompt1, "The prompt was not stored correctly in the request state."
    print(f"✅ Request '{request_id1[:6]}' added successfully.")

    # Check if it was passed to the core engine
    assert len(core._requests_to_add) == 1, f"Expected 1 request in core engine queue, but found {len(core._requests_to_add)}."
    core_req = core._requests_to_add[0]
    assert core_req.request_id == request_id1, "Request ID mismatch between UserLLMEngine and MiniEngineCore."
    assert core_req.prompt == prompt1, "Prompt mismatch between UserLLMEngine and MiniEngineCore."
    print("✅ Request successfully passed to MiniEngineCore.")

    # Test adding a second request
    prompt2 = "The capital of France is"
    request_id2 = engine.add_request(prompt2)
    assert request_id2 in engine.requests, "Second request was not added correctly."
    assert len(core._requests_to_add) == 2, "Core engine queue should have 2 requests after adding another."
    print("✅ Second request added successfully.")
    print("--- Test for Exercise 1 Passed! ---\n")

# Run the test
asyncio.run(test_add_request())

### Exercise 2: Implementing the Streaming `generate` Method

Great job! Now for the most important part: implementing the `generate` method. This method will provide the streaming experience to the user.

It needs to be an `async def` function that uses `yield` to return tokens as they become available, making it an *asynchronous generator*.

Here's the logic you need to implement inside `generate`:

1.  **Add the request**: Call the `add_request` method you just wrote to get a `request_id`.
2.  **Start a loop**: Continuously check for new outputs from the core engine. A `while True` loop is appropriate here.
3.  **Drive the core engine**: Inside the loop, `await` a call to `self.engine_core.step()`. This simulates one processing cycle and returns a list of `RequestOutput` objects for all active requests. Add these new outputs to the `UserLLMEngine`'s `output_queue`.
4.  **Process the queue**: Check your local `output_queue`. For each item, see if its `request_id` matches the ID of the *current* `generate` call.
    *   If it matches, `yield` the token.
    *   If the output is marked as `finished`, you've reached the end of the sequence for this request. Break the `while` loop.
5.  **Clean up**: After the loop finishes, remove the request's state from the `self.requests` dictionary to prevent memory leaks.

This structure ensures that each call to `generate` only yields tokens for its own request, even if the core engine is processing many requests concurrently.

In [ ]:
# GRADED FUNCTION: UserLLMEngine (Part 2)

class UserLLMEngine(UserLLMEngine): # Extends the class from the previous cell
    async def generate(self, prompt: str) -> AsyncIterator[str]:
        """
        Adds a request and yields generated tokens as they become available.

        Args:
            prompt: The user's prompt.
        
        Yields:
            Tokens of the generated text.
        """
        request_id = self.add_request(prompt)
        
        try:
            while True:
                # Drive the engine forward by one step and get all outputs
                step_outputs = await self.engine_core.step()
                self.output_queue.extend(step_outputs)

                # Process our local queue to find outputs for *this* request
                processed_in_this_iteration = False
                temp_queue = deque() # To hold items not for us
                
                while self.output_queue:
                    output = self.output_queue.popleft()
                    
                    if output.request_id == request_id:
                        ### START CODE HERE ###
    # your code here
    pass
                        ### END CODE HERE ###
                    else:
                        # This output is for a different request, put it back for later.
                        temp_queue.append(output)

                self.output_queue.extendleft(reversed(temp_queue)) # Preserve order
                
                if processed_in_this_iteration and output.request_id == request_id and output.finished:
                    break # Exit the outer while loop
                    
        finally:
            # Important: Clean up the request state after we are done.
            if request_id in self.requests:
                del self.requests[request_id]

In [ ]:
# Test for Exercise 2

async def test_generate():
    print("--- Running Test for Exercise 2 ---")
    core = MiniEngineCore()
    engine = UserLLMEngine(engine_core=core)
    
    prompt = "Hello"
    expected_tokens = [" My", " name", " is", " AI", "."]
    
    # Collect all tokens from the async generator
    print(f"Generating for prompt: '{prompt}'")
    start_time = time.time()
    generated_tokens = [token async for token in engine.generate(prompt)]
    end_time = time.time()
    
    print(f"Received tokens: {generated_tokens}")
    print(f"Generation took {end_time - start_time:.2f} seconds.")
    
    assert generated_tokens == expected_tokens, f"Generated tokens do not match expected. Got {generated_tokens}, expected {expected_tokens}"
    print("✅ Token sequence is correct.")
    
    assert not engine.requests, f"Engine should have no active requests after generation, but found {len(engine.requests)}."
    print("✅ Request state was cleaned up correctly.")
    print("--- Test for Exercise 2 Passed! ---\n")

# Run the test
asyncio.run(test_generate())

### Optional Challenge: Handling Concurrent Requests

Your `generate` method is designed to be robust against concurrent requests. Even though the `MiniEngineCore` interleaves the tokens it produces for different requests, each `generate` call should correctly filter and yield only the tokens meant for it.

This final test cell will verify this behavior. We will use `asyncio.gather` to run two `generate` calls concurrently. The test will pass if both streams receive their complete and correct sequence of tokens, demonstrating that your implementation works as intended in a multi-user scenario.

No new code is required from you for this part, but understanding why this works is key to understanding the asynchronous API pattern.

In [ ]:
# Integration Test: Concurrent Generation

async def run_concurrent_test():
    print("--- Running Integration Test: Concurrent Generation ---")
    core = MiniEngineCore()
    engine = UserLLMEngine(engine_core=core)

    prompt1 = "Hello"
    expected1 = [" My", " name", " is", " AI", "."]
    
    prompt2 = "The capital of France is"
    expected2 = [" Par", "is", "."]

    # Helper function to run generation and collect tokens
    async def collect_tokens(prompt):
        result = []
        print(f"Starting generation for: '{prompt}'")
        async for token in engine.generate(prompt):
            print(f"  -> Received for '{prompt}': '{token}'")
            result.append(token)
            await asyncio.sleep(0.02) # Simulate client-side processing
        return result

    # Run both generation tasks concurrently
    results = await asyncio.gather(
        collect_tokens(prompt1),
        collect_tokens(prompt2)
    )

    generated1, generated2 = results
    
    print("\n--- Results ---")
    print(f"Prompt 1 ('{prompt1}') -> Generated: {''.join(generated1)}")
    print(f"Prompt 2 ('{prompt2}') -> Generated: {''.join(generated2)}")

    assert generated1 == expected1, f"Mismatch for prompt 1. Got {generated1}"
    assert generated2 == expected2, f"Mismatch for prompt 2. Got {generated2}"
    print("\n✅ Both concurrent requests generated the correct token sequences.")

    assert not engine.requests, "Engine should have no active requests after all concurrent generations are done."
    print("✅ Engine state is clean after concurrent execution.")
    print("--- Integration Test Passed! ---")

# Run the integration test
asyncio.run(run_concurrent_test())
print("\nCongratulations! You've successfully implemented a user-facing streaming LLM engine.")